# 🧪 Document Classification Pipeline Analysis

This notebook provides a tool for detailed inspection of the classification graph. You can:
1. **Skip OCR** by providing text directly.
2. **Analyze Logprobs** and token distribution for each node.
3. **Debug Decision Margins** and HITL triggers.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import json
import asyncio
from pathlib import Path
from dotenv import load_dotenv

# Force load .env from root
load_dotenv("../.env") 

from config.settings import AppConfig
from graph.builder import build_classification_graph
from graph.state import create_initial_state

print("✅ Modules loaded successfully")

In [ ]:
# 1. Initialize Config and Graph
config = AppConfig()
graph = build_classification_graph(config, use_checkpointer=False)

print(f"📌 Current Model: {'OpenAI (' + config.openai_model + ')' if config.openai_api_key else 'Azure OpenAI'}")
print(f"📌 Margin Threshold: {config.margin_threshold}")

### 📊 Analysis Helper
This function prints the logprob distribution captured at each node.

In [ ]:
def print_node_analysis(node_name, logprobs, margin, threshold):
    if not logprobs:
        print(f"\n❌ No logprob data found for {node_name}")
        return
        
    print(f"\n--- {node_name.upper()} ANALYSIS ---")
    print(f"Winning Token: {logprobs.top1_token} ({logprobs.top1_logprob:.4f})")
    print(f"Runner-up:     {logprobs.top2_token} ({logprobs.top2_logprob:.4f})")
    
    status = "✅ SAFE" if margin >= threshold else "⚠️ UNCERTAIN (HITL)"
    print(f"Margin Score:  {margin:.4f} (Threshold: {threshold}) -> {status}")
    print(f"Confidence:    {logprobs.confidence_pct:.2f}%")

async def run_pipeline(text=None, file_path="dummy.pdf"):
    state = create_initial_state(
        document_id="notebook_test",
        file_path=file_path,
        file_type="pdf"
    )
    
    if text:
        state["azure_ocr_text"] = text
        print("🚀 Running with provided text (skipping OCR)...\n")
    else:
        print(f"🚀 Running with file: {file_path}...\n")
        
    final_state = await graph.ainvoke(state)
    
    # 1. Root Stage
    print_node_analysis("root", final_state["root_logprobs"], final_state["root_margin"], config.margin_threshold)
    
    # 2. Sub Stage
    if final_state.get("sub_code"):
        print_node_analysis("specialist", final_state["sub_logprobs"], final_state["sub_margin"], config.margin_threshold)
        print(f"\nFinal Result: {final_state['root_code']} -> {final_state['sub_code']}")
    else:
        print(f"\nFinal Result: {final_state['root_code']} (Terminated early)")
        
    print(f"Trail: {' -> '.join(final_state['execution_trail'])}")
    return final_state

### 🧪 Scenario 1: Medical Lab Result (Clear)

In [ ]:
test_text = """
PATIENT NAME: JOHN DOE
LABORATORY REPORT
TEST: HAEMOGLOBIN A1C
RESULT: 5.7%
REFERENCE: < 6.0%
"""

result = await run_pipeline(text=test_text)

### 🧪 Scenario 2: Ambiguous Document (Low Margin)
Try a document that looks like both or neither.

In [ ]:
ambiguous_text = """
This is a letter about a claim. It mentions a doctor but looks like an invoice.
Please check the amount of $500 for medical consultation.
"""

result = await run_pipeline(text=ambiguous_text)